In [10]:
import numpy as np
import pandas as pd
import pymongo
import sklearn
import matplotlib
import seaborn as sns

print('All required libraries are installed!')
print('pandas:', pd.__version__)
print('NumPy:', np.__version__)
print('PyMongo:', pymongo.version)
print('scikit-learn:', sklearn.__version__)
print('Matplotlib:', matplotlib.__version__)
print('Seaborn:', sns.__version__)

All required libraries are installed!
pandas: 2.2.3
NumPy: 2.2.4
PyMongo: 4.19.0
scikit-learn: 1.6.1
Matplotlib: 3.10.1
Seaborn: 0.13.2


In [11]:

# CELL 1: LOAD AND VALIDATE THE DATASET

import json
import math
from pathlib import Path
from pprint import pprint
from collections import Counter
from datetime import datetime, timezone

import pandas as pd
from IPython.display import display

# Locate the project directory
DATASET_NAME = "bugradar issues og.json"
project_dir = Path.cwd()

while not (project_dir / "data" / DATASET_NAME).is_file():
    if project_dir.parent == project_dir:
        raise FileNotFoundError(
            f"Cannot find data/{DATASET_NAME}. "
            "Open the BugRadar project folder in VS Code."
        )
    project_dir = project_dir.parent

source_path = project_dir / "data" / DATASET_NAME

# Load the original JSON without modifying it
with source_path.open("r", encoding="utf-8") as file:
    original_data = json.load(file)

# Validate the MongoDB-style document structure
if not isinstance(original_data, list) or not original_data:
    raise ValueError("Expected a non-empty list of documents.")

if not all(isinstance(doc, dict) for doc in original_data):
    raise ValueError("Every record must be a JSON document/object.")

# Get all top-level fields across the documents
all_top_level_fields = sorted({
    key for doc in original_data for key in doc
})

print("Dataset loaded successfully!")
print("Total documents:", len(original_data))
print("Distinct top-level fields:", len(all_top_level_fields))
print("Dataset path:", source_path)


Dataset loaded successfully!
Total documents: 5000
Distinct top-level fields: 20
Dataset path: c:\Users\riya choudhary\OneDrive\Desktop\BugRadar\data\bugradar issues og.json


In [12]:

# CELL 2: INSPECT DOCUMENT STRUCTURE

# Display one complete document with its nested fields
print("SAMPLE MONGODB-STYLE DOCUMENT")
pprint(original_data[0], sort_dicts=False, width=100)

# Inspect nested documents and arrays
nested_fields = [
    "user", "repository", "environment",
    "milestone", "labels", "assignees"
]

structure = []

for field in nested_fields:
    present_values = [
        doc[field] for doc in original_data if field in doc
    ]

    structure.append({
        "Field": field,
        "Present": len(present_values),
        "Absent": len(original_data) - len(present_values),
        "Value types": dict(
            Counter(type(value).__name__ for value in present_values)
        )
    })

display(pd.DataFrame(structure).set_index("Field"))

print("\nTop-level fields:")
for field in all_top_level_fields:
    print("-", field)


SAMPLE MONGODB-STYLE DOCUMENT
{'id': 1000000,
 'number': 1,
 'title': 'Example missing for refresh the page',
 'body': None,
 'issue_category': 'documentation',
 'state': 'open',
 'state_reason': None,
 'priority': 'medium',
 'created_at': '2025-08-05T17:32:00Z',
 'closed_at': None,
 'resolution_hours': None,
 'labels': [],
 'comments': 7,
 'user': {'login': None, 'type': None},
 'assignees': [],
 'repository': {'name': 'auth-api', 'language': 'TypeScript', 'type': 'backend', 'stars': 144},
 'environment': {'platform': 'Firefox 131', 'version': '1.8.0'},
 'milestone': {'title': 'v2.0'},
 'url': 'https://example.invalid/auth-api/issues/1',
 'source': 'synthetic_educational_dataset'}


,Present,Absent,Value types
Field,,,
user,5000,0,"{'dict': 4713, 'NoneType': 287}"
repository,5000,0,{'dict': 5000}
environment,5000,0,"{'dict': 4385, 'NoneType': 615}"
milestone,5000,0,"{'dict': 1237, 'NoneType': 3763}"
labels,5000,0,{'list': 5000}
assignees,5000,0,{'list': 5000}



Top-level fields:
- assignees
- body
- closed_at
- comments
- created_at
- environment
- id
- issue_category
- labels
- milestone
- number
- priority
- repository
- resolution_hours
- source
- state
- state_reason
- title
- url
- user


In [13]:

# CELL 3: ANALYZE MISSING VALUES

total_docs = len(original_data)
missing_rows = []

# Check top-level fields directly in the original documents
for field in all_top_level_fields:
    values = [doc[field] for doc in original_data if field in doc]

    absent = sum(field not in doc for doc in original_data)
    nulls = sum(value is None for value in values)
    empty_strings = sum(
        isinstance(value, str) and value == "" for value in values
    )
    whitespace = sum(
        isinstance(value, str) and value != "" and not value.strip()
        for value in values
    )
    empty_arrays = sum(
        isinstance(value, list) and len(value) == 0 for value in values
    )

    missing_rows.append({
        "Field": field,
        "Absent": absent,
        "Null": nulls,
        "Empty strings": empty_strings,
        "Whitespace strings": whitespace,
        "Empty arrays": empty_arrays,
        "Missing/null %": round(
            (absent + nulls) / total_docs * 100, 2
        )
    })

missing_df = pd.DataFrame(missing_rows).set_index("Field")

print("TOP-LEVEL FIELD COMPLETENESS")
display(missing_df.sort_values("Missing/null %", ascending=False))

# Check embedded document availability
print("\nEMBEDDED DOCUMENT AVAILABILITY")

for field in ["user", "repository", "environment", "milestone"]:
    null_count = sum(doc.get(field) is None for doc in original_data)
    absent_count = sum(field not in doc for doc in original_data)
    present_count = sum(isinstance(doc.get(field), dict)
                        for doc in original_data)

    print(
        f"{field}: {present_count} objects, "
        f"{null_count} null, {absent_count} absent"
    )

# Check selected nested child fields
print("\nNESTED FIELD COMPLETENESS")

child_paths = [
    ("user", "login"),
    ("user", "type"),
    ("repository", "name"),
    ("repository", "language"),
    ("repository", "stars"),
    ("environment", "platform"),
    ("environment", "version"),
    ("milestone", "title")
]

child_rows = []

for parent, child in child_paths:
    absent_parent = null_parent = absent_child = null_child = 0

    for doc in original_data:
        if parent not in doc:
            absent_parent += 1
        elif doc[parent] is None:
            null_parent += 1
        elif isinstance(doc[parent], dict):
            if child not in doc[parent]:
                absent_child += 1
            elif doc[parent][child] is None:
                null_child += 1

    child_rows.append({
        "Nested field": f"{parent}.{child}",
        "Parent absent": absent_parent,
        "Parent null": null_parent,
        "Child absent": absent_child,
        "Child null": null_child
    })

display(pd.DataFrame(child_rows).set_index("Nested field"))


TOP-LEVEL FIELD COMPLETENESS


,Absent,Null,Empty strings,Whitespace strings,Empty arrays,Missing/null %
Field,,,,,,
milestone,0,3763,0,0,0,75.26
state_reason,0,2055,0,0,0,41.10
closed_at,0,1099,0,0,0,21.98
resolution_hours,0,1099,0,0,0,21.98
environment,0,615,0,0,0,12.30
body,0,372,0,0,0,7.44
user,0,287,0,0,0,5.74
comments,0,127,0,0,0,2.54
priority,0,60,72,0,0,1.20



EMBEDDED DOCUMENT AVAILABILITY
user: 4713 objects, 287 null, 0 absent
repository: 5000 objects, 0 null, 0 absent
environment: 4385 objects, 615 null, 0 absent
milestone: 1237 objects, 3763 null, 0 absent

NESTED FIELD COMPLETENESS


,Parent absent,Parent null,Child absent,Child null
Nested field,,,,
user.login,0,287,0,643
user.type,0,287,0,1504
repository.name,0,0,0,0
repository.language,0,0,0,782
repository.stars,0,0,0,0
environment.platform,0,615,0,417
environment.version,0,615,0,493
milestone.title,0,3763,0,0


In [14]:

# CELL 4: DETECT DUPLICATES

# Convert nested values to stable JSON strings for comparison
def stable_json(value):
    return json.dumps(
        value, sort_keys=True, ensure_ascii=False
    )

# Check duplicate application-level IDs and issue numbers
for field in ["id", "number"]:
    values = [
        stable_json(doc[field])
        for doc in original_data
        if field in doc and doc[field] is not None
    ]

    counts = Counter(values)
    duplicate_groups = {
        value: count for value, count in counts.items()
        if count > 1
    }

    extra_records = sum(
        count - 1 for count in duplicate_groups.values()
    )

    print(f"Duplicate {field} groups:", len(duplicate_groups))
    print(f"Extra records with repeated {field}:", extra_records)

# Check exact duplicate documents
document_strings = [
    stable_json(doc) for doc in original_data
]

document_counts = Counter(document_strings)
exact_duplicates = {
    doc: count for doc, count in document_counts.items()
    if count > 1
}

print("\nExact duplicate document groups:", len(exact_duplicates))
print(
    "Extra exact duplicate documents:",
    sum(count - 1 for count in exact_duplicates.values())
)

# Check repeated labels inside each document
repeated_label_docs = 0
repeated_label_entries = 0

for doc in original_data:
    labels = doc.get("labels")

    if not isinstance(labels, list):
        continue

    label_names = [
        stable_json(label.get("name", label))
        if isinstance(label, dict) else stable_json(label)
        for label in labels
    ]

    extra = len(label_names) - len(set(label_names))

    if extra > 0:
        repeated_label_docs += 1
        repeated_label_entries += extra

print("Documents with repeated labels:", repeated_label_docs)
print("Extra repeated label entries:", repeated_label_entries)


Duplicate id groups: 0
Extra records with repeated id: 0
Duplicate number groups: 0
Extra records with repeated number: 0

Exact duplicate document groups: 0
Extra exact duplicate documents: 0
Documents with repeated labels: 1715
Extra repeated label entries: 1913


In [15]:

# CELL 5: CHECK INCONSISTENT FORMATS

# Inspect distinct values in important categorical fields
for field in ["issue_category", "priority", "state"]:
    print(f"\nVALUES IN {field.upper()}")

    counts = Counter(
        stable_json(doc[field])
        for doc in original_data
        if field in doc
    )

    display(pd.DataFrame([
        {"Value": value, "Count": count}
        for value, count in counts.items()
    ]).sort_values("Count", ascending=False))

# Count capitalization and whitespace inconsistencies
print("\nSTRING CONSISTENCY CHECK")

for field in ["issue_category", "priority", "state"]:
    values = [
        doc[field] for doc in original_data
        if isinstance(doc.get(field), str)
    ]

    normalized_groups = {}

    for value in values:
        normalized_groups.setdefault(
            value.strip().casefold(), set()
        ).add(value)

    variants = {
        key: sorted(items)
        for key, items in normalized_groups.items()
        if len(items) > 1
    }

    whitespace_count = sum(
        value != value.strip() for value in values
    )

    print(f"\n{field}:")
    print("Values with leading/trailing whitespace:", whitespace_count)
    print("Case/whitespace variant groups:", variants)

# Check missing or blank text
print("\nTITLE AND BODY COMPLETENESS")

for field in ["title", "body"]:
    absent = sum(field not in doc for doc in original_data)
    nulls = sum(doc.get(field) is None for doc in original_data)
    blank = sum(
        isinstance(doc.get(field), str)
        and not doc[field].strip()
        for doc in original_data
    )

    print(
        f"{field}: absent={absent}, null={nulls}, blank={blank}"
    )

# Check data types of key scalar fields
print("\nSCALAR FIELD DATA TYPES")

for field in ["id", "comments", "resolution_hours"]:
    types = Counter(
        type(doc[field]).__name__
        for doc in original_data
        if field in doc and doc[field] is not None
    )

    print(f"{field}: {dict(types)}")



VALUES IN ISSUE_CATEGORY


,Value,Count
3,"""bug""",2749
2,"""enhancement""",937
0,"""documentation""",640
4,"""question""",443
6,"""Bug""",76
7,"""BUG""",56
1,"""Enhancement""",29
9,"""ENHANCEMENT""",24
5,"""Documentation""",16
8,"""QUESTION""",15



VALUES IN PRIORITY


,Value,Count
0,"""medium""",2466
1,"""low""",1192
3,"""high""",833
2,"""critical""",254
7,"""""",72
5,"""HIGH""",71
6,null,60
4,"""Medium""",52



VALUES IN STATE


,Value,Count
1,"""closed""",3901
0,"""open""",1099



STRING CONSISTENCY CHECK

issue_category:
Values with leading/trailing whitespace: 0
Case/whitespace variant groups: {'documentation': ['DOCUMENTATION', 'Documentation', 'documentation'], 'enhancement': ['ENHANCEMENT', 'Enhancement', 'enhancement'], 'bug': ['BUG', 'Bug', 'bug'], 'question': ['QUESTION', 'Question', 'question']}

priority:
Values with leading/trailing whitespace: 0
Case/whitespace variant groups: {'medium': ['Medium', 'medium'], 'high': ['HIGH', 'high']}

state:
Values with leading/trailing whitespace: 0
Case/whitespace variant groups: {}

TITLE AND BODY COMPLETENESS
title: absent=0, null=0, blank=0
body: absent=0, null=372, blank=0

SCALAR FIELD DATA TYPES
id: {'int': 5000}
comments: {'int': 4743, 'str': 130}
resolution_hours: {'float': 3896, 'int': 5}


In [16]:

# CELL 6: VALIDATE TIMESTAMPS AND RESOLUTION TIME

def parse_timestamp(value):
    """Parse a timestamp for validation without changing the source."""
    if not isinstance(value, str) or not value.strip():
        return None

    try:
        parsed = datetime.fromisoformat(
            value.strip().replace("Z", "+00:00")
        )

        if parsed.tzinfo is None:
            parsed = parsed.replace(tzinfo=timezone.utc)

        return parsed

    except (ValueError, TypeError, OverflowError):
        return None


timestamp_counts = Counter()
closed_before_created = 0
open_with_close = 0
closed_without_close = 0
missing_resolution = 0
invalid_resolution = 0
negative_resolution = 0
duration_mismatches = 0
comparable_durations = 0

for doc in original_data:
    created_raw = doc.get("created_at")
    closed_raw = doc.get("closed_at")

    created = parse_timestamp(created_raw)
    closed = parse_timestamp(closed_raw)

    # Record timestamp issues
    for field, raw, parsed in [
        ("created_at", created_raw, created),
        ("closed_at", closed_raw, closed)
    ]:
        if field not in doc:
            timestamp_counts[f"{field}: absent"] += 1
        elif raw is None:
            timestamp_counts[f"{field}: null"] += 1
        elif parsed is None:
            timestamp_counts[f"{field}: invalid"] += 1

    state = doc.get("state")
    state = state.strip().casefold() if isinstance(state, str) else None

    has_close_value = (
        closed_raw is not None
        and not (
            isinstance(closed_raw, str) and not closed_raw.strip()
        )
    )

    if created is not None and closed is not None:
        if closed < created:
            closed_before_created += 1

    if state == "open" and has_close_value:
        open_with_close += 1

    if state == "closed" and not has_close_value:
        closed_without_close += 1

    # Validate stored resolution_hours
    resolution = doc.get("resolution_hours")

    if resolution is None:
        missing_resolution += 1
        continue

    if (
        isinstance(resolution, bool)
        or not isinstance(resolution, (int, float))
        or not math.isfinite(resolution)
    ):
        invalid_resolution += 1
        continue

    if resolution < 0:
        negative_resolution += 1
        continue

    # Compare with elapsed time when both timestamps are valid
    if created is not None and closed is not None and closed >= created:
        calculated = (closed - created).total_seconds() / 3600
        comparable_durations += 1

        # Stored values are rounded to 2 decimal places
        if abs(float(resolution) - calculated) > 0.005001:
            duration_mismatches += 1

print("TIMESTAMP VALIDATION")
for issue, count in timestamp_counts.items():
    print(f"{issue}: {count}")

print("\nRESOLUTION VALIDATION")
print("Closing date before creation date:", closed_before_created)
print("Open issues with closing timestamp:", open_with_close)
print("Closed issues without closing timestamp:", closed_without_close)
print("Missing resolution_hours:", missing_resolution)
print("Invalid resolution_hours:", invalid_resolution)
print("Negative resolution_hours:", negative_resolution)
print("Comparable durations:", comparable_durations)
print("Duration mismatches beyond rounding tolerance:", duration_mismatches)


TIMESTAMP VALIDATION
closed_at: null: 1099
closed_at: invalid: 22
created_at: null: 30

RESOLUTION VALIDATION
Closing date before creation date: 0
Open issues with closing timestamp: 0
Closed issues without closing timestamp: 0
Missing resolution_hours: 1099
Invalid resolution_hours: 0
Negative resolution_hours: 0
Comparable durations: 3857
Duration mismatches beyond rounding tolerance: 0


In [18]:

# CELL 7: DATA-QUALITY SUMMARY

print("=" * 50)
print("BUGRADAR — DATASET EXPLORATION SUMMARY")
print("=" * 50)

print("Total documents:", len(original_data))
print("Top-level fields:", len(all_top_level_fields))

print(
    "Documents with repeated labels:",
    repeated_label_docs
)

print(
    "Exact duplicate document groups:",
    len(exact_duplicates)
)

print(
    "Issues closed before creation:",
    closed_before_created
)

print(
    "Open issues with closing timestamps:",
    open_with_close
)

print(
    "Closed issues without closing timestamps:",
    closed_without_close
)

print(
    "Resolution duration mismatches:",
    duration_mismatches
)



BUGRADAR — DATASET EXPLORATION SUMMARY
Total documents: 5000
Top-level fields: 20
Documents with repeated labels: 1715
Exact duplicate document groups: 0
Issues closed before creation: 0
Open issues with closing timestamps: 0
Closed issues without closing timestamps: 0
Resolution duration mismatches: 0
